# Calibrate LeakyBucket model using scipy bounded optimisation

Calibrates the single `leakiness` parameter of the LeakyBucket model against
ERA5 forcing and Caravan observed discharge over the calibration period defined
in `settings.json`.

**Objective:** maximise KGE (Kling-Gupta Efficiency) → minimise `1 - KGE`.

**Optimiser:** `scipy.optimize.minimize_scalar` with `method='bounded'`.
One parameter, no heavy dependencies — no `sceua` needed.

**Output:**
- `{path_output}/{caravan_id}_params_leakybucket.csv` — best leakiness value
- `results.json` — calibration KGE and NSE

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

from rich import print

import ewatercycle
import ewatercycle.forcing

try:
    from scripts.leakybucket_model import LeakyBucketLocal
except ImportError:
    sys.path.insert(0, str(Path().resolve().parent))
    from scripts.leakybucket_model import LeakyBucketLocal

In [ ]:
# Parameters — overridden by papermill when running on HPC
settings_path = "settings.json"

In [ ]:
with open(settings_path, 'r') as f:
    settings = json.load(f)

display(settings)

In [ ]:
# Skip if calibration output already exists
params_path = Path(settings['path_output']) / (settings['caravan_id'] + '_params_leakybucket.csv')
need_to_run = not params_path.exists()

if not need_to_run:
    display(f'Calibration already complete: {params_path}')

## Load forcing and observations

In [ ]:
# ERA5 forcing (LumpedMakkink) — used as model input
era5_dir = Path(settings['path_ERA5']) / 'work' / 'diagnostic' / 'script'
era5_forcing = ewatercycle.forcing.sources['LumpedMakkinkForcing'].load(directory=era5_dir)
display(era5_forcing)

In [ ]:
# Caravan observed discharge
caravan_forcing = ewatercycle.forcing.sources['CaravanForcing'].load(
    directory=settings['path_caravan']
)
obs_q = xr.open_dataset(
    caravan_forcing.directory / caravan_forcing['Q']
)['Q'].to_series()
obs_q.index = obs_q.index.tz_localize(None)
print(f'Observed discharge: {len(obs_q)} days, '
      f'{obs_q.index[0].date()} → {obs_q.index[-1].date()}')

## Calibration objective and model runner

In [ ]:
def kge(sim: np.ndarray, obs: np.ndarray) -> float:
    """Kling-Gupta Efficiency."""
    mask = ~(np.isnan(sim) | np.isnan(obs))
    s, o = sim[mask], obs[mask]
    if o.std() == 0 or s.std() == 0:
        return -999.0
    r = np.corrcoef(s, o)[0, 1]
    alpha = s.std() / o.std()
    beta  = s.mean() / o.mean()
    return float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))


def nse(sim: np.ndarray, obs: np.ndarray) -> float:
    """Nash-Sutcliffe Efficiency."""
    mask = ~(np.isnan(sim) | np.isnan(obs))
    s, o = sim[mask], obs[mask]
    return float(1 - np.sum((s - o)**2) / np.sum((o - o.mean())**2))

In [ ]:
def run_leakybucket(leakiness: float, forcing, start: str, end: str) -> pd.Series:
    """Run LeakyBucketLocal and return a discharge time series."""
    model = LeakyBucketLocal(forcing=forcing)
    cfg_file, _ = model.setup(
        leakiness=leakiness,
        initial_storage=0.0,
        start_time=start,
        end_time=end,
    )
    model.initialize(cfg_file)

    q_vals, timestamps = [], []
    while model.time < model.end_time:
        model.update()
        q_vals.append(model.get_value('discharge', dest=np.zeros(1))[0])
        timestamps.append(pd.Timestamp(model.time_as_datetime))

    model.finalize()
    return pd.Series(data=q_vals, index=pd.DatetimeIndex(timestamps).tz_localize(None))


def objective(leakiness: float) -> float:
    """Return 1 − KGE for the calibration period (scipy minimises)."""
    sim = run_leakybucket(
        leakiness,
        era5_forcing,
        settings['calibration_start_date'],
        settings['calibration_end_date'],
    )
    obs = obs_q.reindex(sim.index)
    score = 1.0 - kge(sim.values, obs.values)
    return score

## Run optimisation

In [ ]:
if need_to_run:
    print('Calibrating LeakyBucket — searching leakiness in [0.001, 0.5] ...')

    result = minimize_scalar(
        objective,
        bounds=(0.001, 0.5),
        method='bounded',
        options={'xatol': 1e-4, 'maxiter': 100},
    )

    best_leakiness = result.x
    best_kge       = 1.0 - result.fun

    print(f'Best leakiness = [bold cyan]{best_leakiness:.6f}[/bold cyan]  '
          f'KGE = [bold green]{best_kge:.4f}[/bold green]  '
          f'(iterations: {result.nit})')

In [ ]:
if need_to_run:
    # Re-run with best parameters to get the full discharge series
    sim_cal = run_leakybucket(
        best_leakiness,
        era5_forcing,
        settings['calibration_start_date'],
        settings['calibration_end_date'],
    )
    obs_cal = obs_q.reindex(sim_cal.index)

    best_nse = nse(sim_cal.values, obs_cal.values)
    print(f'Calibration period — KGE: {best_kge:.4f}, NSE: {best_nse:.4f}')

## Calibration plot

In [ ]:
if need_to_run:
    zoom_start = pd.Timestamp(settings['calibration_start_date'][:10])
    zoom_end   = zoom_start + pd.DateOffset(years=2)

    fig, ax = plt.subplots(figsize=(12, 4))
    obs_cal.loc[zoom_start:zoom_end].plot(ax=ax, color='black', lw=0.9, label='Observed (Caravan)')
    sim_cal.loc[zoom_start:zoom_end].plot(ax=ax, color='steelblue', lw=1.4, label=f'LeakyBucket (leakiness={best_leakiness:.4f})')
    ax.set_ylabel('Discharge (m d⁻¹)')
    ax.set_title(f'LeakyBucket calibration — {settings["caravan_id"]}  '
                 f'(2-year zoom)\nKGE={best_kge:.3f}, NSE={best_nse:.3f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    fig_path = Path(settings['figure_output']) / 'calibration_leakybucket.png'
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()

## Save results

In [ ]:
if need_to_run:
    # Save best leakiness to CSV
    params_path.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(params_path, [best_leakiness], delimiter=',')
    print(f'Saved leakiness={best_leakiness:.6f} → {params_path}')

In [ ]:
if need_to_run:
    # Write calibration metrics to results.json
    results_path = Path(settings['results_path'])
    results = json.loads(results_path.read_text()) if results_path.exists() else {}

    results.setdefault('caravan_id', settings['caravan_id'])
    results['calibration_leakybucket'] = {
        'KGE':    round(best_kge, 4),
        'NSE':    round(best_nse, 4),
        'leakiness': round(best_leakiness, 6),
        'period': f"{settings['calibration_start_date'][:10]} / {settings['calibration_end_date'][:10]}",
    }

    results_path.write_text(json.dumps(results, indent=4))
    print(f"Calibration KGE={results['calibration_leakybucket']['KGE']}, "
          f"NSE={results['calibration_leakybucket']['NSE']} → {results_path}")